# Colab learning budget preparation

This notebook prepares a predeclared development learning-budget study with independent User Vehicle decisions, the unchanged cost/physical models, and standard QR/QR-CVaR training. It is not main training or a calibrated paper benchmark. The logarithmic and legacy SNR encoders remain explicit variants; this notebook does not select a winning variant or checkpoint.

**All execution gates default to False. Run all cannot mount Drive or start training.** After verified source setup, preflight and the planner only inspect runtime/protocol information. The run cell advances one manually selected model chain, never the whole grid. This driver uses CPU; selecting a Colab GPU does not accelerate it.

## Prepare the exact private source

After this milestone is committed and pushed, obtain the full commit and Python-source digest from that same trusted checkout:

```text
git rev-parse HEAD
python -c "from experiments.provenance import source_provenance; print(source_provenance()['source_sha256'])"
```

While signed in to GitHub **in your browser**, download `https://github.com/Amirr-1378/risk-aware-vehicular-edge-computing/archive/<FULL_COMMIT>.zip`. Upload that ZIP through Colab's Files sidebar and set `ARCHIVE_PATH` to its uploaded path, or rename it to `source.zip`. Copy the independently obtained commit and digest into the configuration below. GitHub credentials remain in the browser; do not put a password or token in notebook cells, URLs, environment variables, or outputs.

The archive root is checked against the expected commit; the root name alone does not prove Git history. The independently trusted digest checks Python files under `src/` and directly under `experiments/`, with normalized text newlines. Configurations, documentation and `pyproject.toml` are outside that digest: obtain the entire archive from the trusted browser download. The runner separately binds the exact scientific protocols. Start a fresh runtime when changing source versions, and use a new source destination; existing extracted source is never overwritten.


In [ ]:
# Fill identity from the same trusted committed checkout. No credentials belong here.
EXPECTED_COMMIT = ""
EXPECTED_SOURCE_SHA256 = ""
ARCHIVE_PATH = "/content/source.zip"
SOURCE_DESTINATION = "/content/risk-aware-v2x-budget-source"

# A new root isolates this study from earlier development sessions.
STUDY_OUTPUT_ROOT = "/content/drive/MyDrive/risk-aware-v2x/learning-budget-v1"
VARIANT = "log_snr_v4"
ALGORITHM = "qr_dqn_cvar"
AGENT_SEED = 11
MAX_ADDITIONAL_EPISODES = 10
RESUME = False

# Every action gate remains opt-in; configuration resets readiness.
RUN_TESTS = False
MOUNT_DRIVE = False
RUN_STUDY = False
PREFLIGHT_READY = False
PLAN_READY = False
DRIVE_READY = False

## Verify and extract source locally

The following bootstrap embeds the repository's standalone, standard-library ZIP verifier verbatim, since the source tree does not exist yet. It validates every archive member and the source digest before extraction. It makes no network request and installs no packages. Empty identity placeholders deliberately stop setup.


In [ ]:
from pathlib import Path
import os
import sys

SOURCE_HELPER_CODE = "\"\"\"Validate and extract a browser-downloaded private GitHub source ZIP offline.\n\nThe archive root declares the expected commit; it does not cryptographically\nprove Git history. A separately trusted SHA256 verifies the Python source set\nusing the same text normalization as ``source_provenance``. No credentials,\nnetwork requests, dependency installation or training are involved.\n\"\"\"\n\nimport argparse\nimport hashlib\nimport json\nimport re\nimport stat\nfrom pathlib import Path, PurePosixPath\nfrom zipfile import ZipFile\n\n\n_MAX_EXTRACTED_BYTES = 100 * 1024 * 1024\n_REQUIRED_FILES = frozenset({\"README.md\", \"AGENTS.md\", \"pyproject.toml\",\n    \"src/risk_aware_v2x/contracts.py\", \"experiments/provenance.py\"})\n_WINDOWS_RESERVED = frozenset({\"CON\", \"PRN\", \"AUX\", \"NUL\", \"CONIN$\", \"CONOUT$\",\n    *(f\"COM{i}\" for i in range(1, 10)), *(f\"LPT{i}\" for i in range(1, 10))})\n\n\ndef _member_path(info, expected_root):\n    name = info.filename\n    if \"\\x00\" in info.orig_filename or \"\\\\\" in info.orig_filename or name.startswith(\"/\"):\n        raise ValueError(\"archive contains an unsafe path\")\n    pieces = name.rstrip(\"/\").split(\"/\")\n    if any(not piece or piece in (\".\", \"..\") or any(char in '<>:\"|?*' or ord(char) < 32 for char in piece)\n           or piece.rstrip(\" .\") != piece\n           or piece.split(\".\")[0].upper() in _WINDOWS_RESERVED for piece in pieces):\n        raise ValueError(\"archive contains an unsafe or nonportable path\")\n    if pieces[0] != expected_root:\n        raise ValueError(\"archive must have exactly the expected commit root\")\n    if any(piece.casefold() == \".git\" for piece in pieces[1:]):\n        raise ValueError(\"archive must not contain a .git tree\")\n    mode_type = stat.S_IFMT((info.external_attr >> 16) & 0xFFFF)\n    allowed_type = stat.S_IFDIR if info.is_dir() else stat.S_IFREG\n    if mode_type not in (0, allowed_type):\n        raise ValueError(\"archive members must be regular files or directories\")\n    if info.flag_bits & 1:\n        raise ValueError(\"encrypted archive members are unsupported\")\n    if len(pieces) == 1 and not info.is_dir():\n        raise ValueError(\"archive root must be a directory\")\n    if info.is_dir() and info.file_size:\n        raise ValueError(\"archive directory must not contain file data\")\n    return PurePosixPath(*pieces[1:]).as_posix() if len(pieces) > 1 else \"\"\n\n\ndef _source_sha256(files):\n    digest = hashlib.sha256()\n    for relative in sorted(files):\n        parts = PurePosixPath(relative).parts\n        is_source = relative.endswith(\".py\") and (\n            parts[0] == \"src\" or (parts[0] == \"experiments\" and len(parts) == 2))\n        if is_source:\n            # Path.read_text(encoding='utf-8') uses universal newline handling.\n            text = files[relative].decode(\"utf-8\").replace(\"\\r\\n\", \"\\n\").replace(\"\\r\", \"\\n\")\n            digest.update(relative.encode(\"utf-8\") + b\"\\0\")\n            digest.update(text.encode(\"utf-8\") + b\"\\0\")\n    return digest.hexdigest()\n\n\ndef extract_source_archive(archive: Path, destination: Path, *, expected_commit: str,\n                           expected_source_sha256: str) -> Path:\n    \"\"\"Validate every member and source digest before creating a new root.\n\n    The returned root is ``destination`` with the single GitHub archive prefix\n    stripped. Existing paths are never replaced. The 100 MiB limit applies to\n    the sum of uncompressed member sizes. Non-Python contents are presence/\n    path checked, but are outside the source-provenance SHA256 contract.\n    \"\"\"\n    if not isinstance(expected_commit, str) or not re.fullmatch(r\"[0-9a-fA-F]{40}\", expected_commit):\n        raise ValueError(\"expected_commit must be a full 40-hex commit\")\n    if not isinstance(expected_source_sha256, str) or not re.fullmatch(r\"[0-9a-fA-F]{64}\", expected_source_sha256):\n        raise ValueError(\"expected_source_sha256 must be a trusted 64-hex digest\")\n    destination = Path(destination)\n    if destination.exists() or destination.is_symlink():\n        raise FileExistsError(\"choose a new source destination\")\n    expected_root = \"risk-aware-vehicular-edge-computing-\" + expected_commit\n    files, directories, seen, spellings = {}, set(), set(), {}\n    with ZipFile(archive) as source:\n        members = source.infolist()\n        if sum(member.file_size for member in members) > _MAX_EXTRACTED_BYTES:\n            raise ValueError(\"archive exceeds the 100 MiB extracted-size limit\")\n        for member in members:\n            relative = _member_path(member, expected_root)\n            portable_key = relative.casefold()\n            if portable_key in seen:\n                raise ValueError(\"archive contains duplicate or case-colliding members\")\n            seen.add(portable_key)\n            parts = PurePosixPath(relative).parts\n            for count in range(1, len(parts) + 1):\n                prefix = \"/\".join(parts[:count])\n                key = prefix.casefold()\n                if key in spellings and spellings[key] != prefix:\n                    raise ValueError(\"archive contains case-colliding directory paths\")\n                spellings[key] = prefix\n            if member.is_dir():\n                if relative:\n                    directories.add(relative)\n            else:\n                # Reading all bytes also checks CRC/decompression before any writes.\n                files[relative] = source.read(member)\n        file_keys = {relative.casefold() for relative in files}\n        for relative in (*files, *directories):\n            for parent in PurePosixPath(relative).parents:\n                if parent.as_posix().casefold() in file_keys:\n                    raise ValueError(\"archive file conflicts with a directory\")\n        if not _REQUIRED_FILES.issubset(files):\n            raise ValueError(\"archive is missing required repository files\")\n        if _source_sha256(files) != expected_source_sha256.lower():\n            raise ValueError(\"archive Python sources do not match the trusted source digest\")\n\n    destination.mkdir(parents=True, exist_ok=False)\n    for relative in sorted(directories, key=lambda name: (name.count(\"/\"), name)):\n        destination.joinpath(*PurePosixPath(relative).parts).mkdir(parents=True, exist_ok=True)\n    for relative, data in files.items():\n        target = destination.joinpath(*PurePosixPath(relative).parts)\n        target.parent.mkdir(parents=True, exist_ok=True)\n        with target.open(\"xb\") as output:\n            output.write(data)\n    return destination\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--archive\", type=Path, required=True)\n    parser.add_argument(\"--destination\", type=Path, required=True)\n    parser.add_argument(\"--commit\", required=True)\n    parser.add_argument(\"--source-sha256\", required=True)\n    args = parser.parse_args(argv)\n    root = extract_source_archive(args.archive, args.destination,\n        expected_commit=args.commit, expected_source_sha256=args.source_sha256)\n    print(json.dumps({\"root\": str(root)}))\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n"
bootstrap_namespace = {"__name__": "colab_source_bootstrap"}
exec(compile(SOURCE_HELPER_CODE, "colab_source_bootstrap.py", "exec"), bootstrap_namespace)
REPO = bootstrap_namespace["extract_source_archive"](
    Path(ARCHIVE_PATH), Path(SOURCE_DESTINATION),
    expected_commit=EXPECTED_COMMIT,
    expected_source_sha256=EXPECTED_SOURCE_SHA256,
)
os.chdir(REPO)
for source_path in (str(REPO / "src"), str(REPO)):
    if source_path not in sys.path:
        sys.path.insert(0, source_path)
os.environ["PYTHONPATH"] = os.pathsep.join((str(REPO), str(REPO / "src")))
print(f"Verified source extracted to {REPO}")


## Inspect the runtime before opting in

Preflight only checks installed metadata; it does not import Torch/NumPy, install dependencies, train, evaluate, or inspect test trajectories. It requires Python >=3.11, stable Torch >=2.6,<3, and stable NumPy >=1.26,<3. If checks fail, stop and explicitly choose or prepare a compatible runtime using `pyproject.toml`; there is no automatic installation cell. A later learning import can still expose an environment problem not detectable from package metadata.

Resume additionally requires the original exact source, protocol, encoder, algorithm/seed/scope and runtime identity. Broadly compatible version ranges are insufficient for resuming an existing session. A replacement Colab VM may require explicitly restoring the original runtime rather than continuing with different versions.


In [ ]:
import json
from experiments.runtime_preflight import collect_preflight

PREFLIGHT_READY = False
PREFLIGHT = collect_preflight(require_learning=True)
print(json.dumps(PREFLIGHT, indent=2, allow_nan=False))
if PREFLIGHT["provenance"]["source_sha256"] != EXPECTED_SOURCE_SHA256.lower():
    raise RuntimeError("Extracted/imported Python source does not match the expected digest.")
if not PREFLIGHT["ok"]:
    raise RuntimeError("Runtime preflight failed; prepare compatible dependencies explicitly before continuing.")
PREFLIGHT_READY = True


## Inspect the fixed study plan

After successful source/runtime preflight, this cell loads and validates the checked-in learning-budget protocol and expanded development scenario plan, then displays metadata counts. It constructs no scenario trajectory, trains no model and writes no session output. Loading the protocol can import the installed learning libraries; metadata preflight alone did not prove those imports.

The declared grid has two encoder representations, two algorithms (QR and QR-CVaR) and three predeclared agent seeds: 12 model chains. Each chain has fixed budget points at 20, 100 and 500 training episodes and the same 20 validation cases in each of four known profiles. The full plan has 6000 training executions, 2880 learned validation executions and 400 shared fixed-policy executions: 9280 in total. These counts are a declared plan, not completed results or a wall-time estimate. This notebook never loops over that grid automatically.

Intermediate evaluations are descriptive learning-curve points. Validation cannot extend the budget, choose an early winner or select a best checkpoint. The maximum budget remains fixed before the first run; held-out test trajectories stay closed.

In [ ]:
if not PREFLIGHT_READY:
    raise RuntimeError("Run successful verified-source setup and runtime preflight before planning.")
PLAN_READY = False
import json
from experiments.learning_budget_protocol import (
    DEFAULT_BUDGET_SCENARIO_PROTOCOL, DEFAULT_LEARNING_BUDGET_PROTOCOL, LearningBudgetProtocol,
)
from risk_aware_v2x.scenarios.protocol import DevelopmentProtocol

BUDGET_PROTOCOL = LearningBudgetProtocol.load(DEFAULT_LEARNING_BUDGET_PROTOCOL)
BUDGET_SCENARIOS = DevelopmentProtocol.load(DEFAULT_BUDGET_SCENARIO_PROTOCOL)
BUDGET_PROTOCOL.validate_against(BUDGET_SCENARIOS)
BUDGET_PLAN = BUDGET_PROTOCOL.execution_plan()
print(json.dumps(BUDGET_PLAN, indent=2, allow_nan=False))
PLAN_READY = True

## Optional offline setup checks

Set RUN_TESTS=True in a new control cell to run only this notebook's offline checks, source ZIP validation and metadata preflight tests. Standard-library unittest is sufficient; these tests do not train or mount Drive. The full repository test suite contains small learning fixtures and is not launched here.

In [ ]:
if RUN_TESTS:
    import subprocess
    import sys
    for pattern in ("test_budget_colab_notebook.py", "test_colab_source.py", "test_runtime_preflight.py"):
        subprocess.run([
            sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", pattern, "-v",
        ], cwd=REPO, check=True)
else:
    print("Offline setup checks skipped: RUN_TESTS=False.")

## Explicitly mount output storage

When ready, set MOUNT_DRIVE=True in a new control cell and run the mount cell. Choose a study root inside MyDrive. A model chain gets its own root/variant/algorithm/seed folder, and exactly that folder is reused for resume. Never use two writers for the same chain.

The driver publishes one checkpoint per completed training chunk, normally up to ten episodes, and a separate frozen-validation record at each budget point. More frequent pauses can create more checkpoints. Large sweeps can require substantial Drive storage; the plan counts do not measure available quota or predict artifact byte size. Preserve complete artifacts and wait for synchronization before ending a runtime. Hashes and filesystem publication detect corruption but do not guarantee cloud transaction durability.

In [ ]:
DRIVE_READY = False
if MOUNT_DRIVE:
    from pathlib import Path
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_READY = Path("/content/drive/MyDrive").is_dir()
    if not DRIVE_READY:
        raise RuntimeError("Drive MyDrive is unavailable after mounting.")
else:
    print("Drive not mounted: MOUNT_DRIVE=False.")


## Advance one selected development chain

Preparation does not start this study. To opt in later, add a new control cell such as:

```python
MOUNT_DRIVE = True
RUN_STUDY = True
RESUME = False
MAX_ADDITIONAL_EPISODES = 10
```

Run that control cell, then the Drive mount cell and the selected-chain cell below. **Do not rerun configuration or ZIP extraction**: configuration clears readiness flags, and extraction refuses to replace source. The initial selected chain is log_snr_v4 / qr_dqn_cvar / seed 11. Change the three selection variables explicitly only when starting another declared chain, with RESUME=False for its fresh folder.

Each invocation adds at most the stated number of training episodes to the same fixed chain. The cap counts training episodes only: reaching 20, 100 or 500 additionally runs all 80 fixed frozen-validation episodes before publishing that milestone or continuing training, so those calls can take longer. No manual extension or best-checkpoint selection is supported. Resume reloads the exact model/optimizer/replay/RNG state from its latest committed chunk; an interrupted unpublished chunk repeats the same predeclared scenes from the preceding boundary.

The compact result shows only this chain's progress. Partial chains are incomplete study evidence; no overall grid result or comparison against greedy follows from one chain.

In [ ]:
if RUN_STUDY:
    if not PREFLIGHT_READY:
        raise RuntimeError("Run successful source/runtime preflight first.")
    if not PLAN_READY:
        raise RuntimeError("Inspect and validate the fixed study plan first.")
    if not MOUNT_DRIVE or not DRIVE_READY:
        raise RuntimeError("Explicitly mount Drive and confirm MyDrive before study execution.")
    if VARIANT not in BUDGET_PROTOCOL.variants or ALGORITHM not in BUDGET_PROTOCOL.learning.algorithms:
        raise ValueError("Choose a representation and algorithm declared in the fixed plan.")
    if type(AGENT_SEED) is not int or AGENT_SEED not in BUDGET_PROTOCOL.learning.agent_seeds:
        raise ValueError("Choose an agent seed declared in the fixed plan.")
    from pathlib import Path
    import json
    drive_root = Path("/content/drive/MyDrive")
    study_root = Path(STUDY_OUTPUT_ROOT)
    if (not study_root.is_absolute() or study_root == drive_root
            or not study_root.resolve().is_relative_to(drive_root.resolve())):
        raise ValueError("Choose an absolute study subfolder inside Drive MyDrive.")
    output_path = study_root / VARIANT / ALGORITHM / f"seed-{AGENT_SEED}"
    if not output_path.resolve().is_relative_to(study_root.resolve()):
        raise ValueError("Selected chain output must remain inside the study root.")
    from experiments.run_learning_budget_session import run_learning_budget_session
    STUDY_RESULT = run_learning_budget_session(
        output_directory=output_path,
        protocol=BUDGET_PROTOCOL,
        scenarios=BUDGET_SCENARIOS,
        variant=VARIANT,
        algorithm=ALGORITHM,
        agent_seed=AGENT_SEED,
        resume=RESUME,
        max_additional_episodes=MAX_ADDITIONAL_EPISODES,
    )
    summary_keys = (
        "status", "completed_training_episodes", "planned_training_episodes",
        "completed_budget_points", "planned_budget_points", "latest_validation_budget",
        "completed_validation_episodes", "planned_validation_episodes",
        "training_slots", "seen_transitions", "gradient_updates", "checkpoint", "model_sha256",
        "main_training_started", "test_split_accessed",
    )
    STUDY_SUMMARY = {key: STUDY_RESULT[key] for key in summary_keys}
    print(json.dumps(STUDY_SUMMARY, indent=2, allow_nan=False))
else:
    print("Study execution skipped: RUN_STUDY=False.")

## Continue or recover a chain

For ordinary same-runtime continuation, set RESUME=True in a new cell and rerun only the selected-chain cell. Keep its variant, algorithm, seed, output root and scientific plan unchanged. Restore successful preflight/planning and the same mounted Drive after reconnecting; exact source and runtime identities must match the original chain. A changed Python/Torch/NumPy build or encoder is not an exact resume.

A published chunk ends at complete episodes. If an incomplete final generation remains, the driver fails closed without deleting or overwriting evidence. The existing `experiments.session_journal.copy_verified_prefix(source, fresh_destination, exact_original_contract)` can explicitly recover into a new folder; it is not executed here. Corrupt committed files or an incomplete middle generation cannot be silently discarded.

Full validation records and checkpoint history remain in the immutable journal; STUDY_RESULT retains the most recent result in memory. Completing one chain does not establish a complete study. Fixed-policy references are shared across chains, and a complete comparison must retain every declared representation, algorithm, seed and budget point. Main training remains a separately announced Colab milestone after calibration and a frozen paper protocol; no main-training or held-out-test launch cell is present.